# Eksperimen 1: Tahap-1 SARIMA vs GBM

Membandingkan **SARIMA** (rujukan [7]) melawan **LightGBM** dan **XGBoost** sebagai tahap-1, dengan tahap-2 neural **BiLSTM/Attention/Fourier** yang identik. Protokol apple-to-apple: test 720 hari, window 120 hari, horizon 1/7/30.

In [ ]:
import sys
BASE='/home/mada/Documents/Kuliah/Algorithms and Complexity/02_Kode/hybrid-lgbm-xgb'
sys.path.insert(0,BASE)
import hybrid as H
import numpy as np, pandas as pd
from scipy.stats import boxcox
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
print('Data dimuat:', H.load().shape)

In [ ]:
import sys
BASE='/home/mada/Documents/Kuliah/Algorithms and Complexity/02_Kode/hybrid-lgbm-xgb'
sys.path.insert(0,BASE)
import pandas as pd, re
# Muat hasil fair compare 45 (hash re-run penuh — sangat lambat)
rows=[]
for l in open(f'{BASE}/hasil_ref_vs_gbm.txt'):
    m=re.match(r'\[OK\] (\S+)\+([\S+]+) (PM\d+|NO2) h(\d+): RMSE=([\d.]+)', l)
    if m:
        rows.append(dict(stage1=m.group(1),stage2=m.group(2),
                         pollutant=m.group(3),horizon=int(m.group(4)),
                         rmse=float(m.group(5))))
df=pd.DataFrame(rows)
print(f'Total hasil dimuat: {len(df)}')
print(df.groupby(['stage1','stage2']).size())

In [ ]:
import pandas as pd
piv=df.pivot_table(index=['stage1','stage2'],columns=['pollutant','horizon'],values='rmse')
piv.style.highlight_min(axis=0)

In [ ]:
# Visualisasi: bar chart per polutan untuk BiLSTM
import matplotlib.pyplot as plt
pol='NO2'
sel=df[(df.stage2=='bilstm')&(df.pollutant==pol)]
fig,ax=plt.subplots(figsize=(9,4))
for s1,c in [('sarima','red'),('lightgbm','orange'),('xgboost','green')]:
    d=sel[sel.stage1==s1].set_index('horizon').rmse.sort_index()
    ax.plot(d.index.astype(str),d.values,marker='o',label=s1,color=c)
ax.set_title(f'RMSE per horizon — {pol} (BiLSTM stage-2)'); ax.set_ylabel('RMSE'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()